## 1. Carga de datos inicial

Al principio pensé en cargarlos directamente usando `utf-8`, pero los archivos del Ayuntamiento daban errores de decodificación. Me tomó varios intentos darme cuenta de que la data original venía en `latin-1`. El JSON de libros no tuvo ese problema.

In [ ]:
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore') # Silenciamos warnings deprecados de matplotlib/pandas

DATA_DIR = os.path.join(os.getcwd(), 'data_example')
OUT_DIR  = os.path.join(os.getcwd(), 'data_clean')
os.makedirs(OUT_DIR, exist_ok=True)

df_terrazas = pd.read_csv(os.path.join(DATA_DIR, "Terrazas_202104.csv"), sep=';', encoding="latin-1", low_memory=False)
df_licencias = pd.read_csv(os.path.join(DATA_DIR, "Licencias_Locales_202104.csv"), sep=';', encoding="latin-1", low_memory=False)
df_books = pd.read_json(os.path.join(DATA_DIR, "books.json"), lines=True)

print(f"Cargados — Terrazas: {len(df_terrazas)} | Licencias: {len(df_licencias)} | Libros: {len(df_books)}")

## 2. Análisis Exploratorio (EDA) y Completitud

Originalmente hice un `df.info()`, pero la consola se llenaba de texto y no veía el problema real. Pasé a usar `missingno` para ver los huecos visualmente.

In [ ]:
plt.style.use("seaborn-v0_8-muted")

fig, ax1 = plt.subplots(figsize=(10, 4))
msno.bar(df_licencias, color="#4A90E2", ax=ax1, fontsize=8)
plt.title("AUDITORÍA: COMPLETITUD DE LICENCIAS", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 4))
msno.matrix(df_terrazas, color=(0.2, 0.5, 0.2), sparkline=True, ax=ax2, fontsize=8)
plt.title("PATRONES DE NULOS EN TERRAZAS (MÉTODO MATRIX)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

Aproveché para explorar la distribución de variables clave antes de limpiar, como la superficie solicitada y en qué distritos se concentran las licencias.

In [ ]:
# Casteamos las comas temporales a float para graficar
superficies = pd.to_numeric(df_terrazas['Superficie_ES'].astype(str).str.replace(',', '.'), errors='coerce').dropna()

fig3, ax3 = plt.subplots(figsize=(10, 5))
sns.histplot(superficies, bins=50, kde=True, color='orange', ax=ax3)
plt.title('Distribución de Superficie Escrita de Terrazas (m2)', fontweight="bold")
plt.tight_layout()
plt.show()

# TOP 10 Distritos
top_distritos = df_licencias['desc_distrito_local'].value_counts().head(10).reset_index()
top_distritos.columns = ['Distrito', 'Conteo']

fig4, ax4 = plt.subplots(figsize=(10, 5))
sns.barplot(data=top_distritos, y='Distrito', x='Conteo', color='teal', ax=ax4)
plt.title('TOP 10 DISTRITOS CON MÁS LICENCIAS', fontweight="bold")
plt.tight_layout()
plt.show()

En el archivo JSON de la biblioteca noté algunas diferencias; por ejemplo la cantidad de páginas tienen un salto largo.

In [ ]:
fig5, ax5 = plt.subplots(figsize=(8, 4))
sns.boxplot(x=df_books['pageCount'].dropna(), color='lightgreen', ax=ax5)
plt.title('Rango de Extensión de Libros (Páginas)', fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Limpieza y estandarización de columnas

Tuve que agregar una función reparadora porque había caracteres rotos en el texto de las calles y descripciones (Mojibake). Además, decidí arreglar de raíz las superficies numéricas erradas para que el Data Warehouse en SQLite no fallara luego.

In [ ]:
def limpiar_texto(columna):
    def reparar(val):
        if not isinstance(val, str): return val
        try: 
            return val.encode('latin-1').decode('utf-8')
        except: 
            return val
            
    return columna.astype(str).str.strip().apply(reparar).str.replace(r'\s+', ' ', regex=True)

# Limpiamos el texto base
cols_terr_texto = df_terrazas.select_dtypes(include=['object', 'string']).columns
for col in cols_terr_texto:
    df_terrazas[col] = limpiar_texto(df_terrazas[col])
    
cols_lic_texto = df_licencias.select_dtypes(include=['object', 'string']).columns
for col in cols_lic_texto:
    df_licencias[col] = limpiar_texto(df_licencias[col])

# Las superficies venian con coma decimal, las pasamos a numerico y rellenamos nulos con 0
cols_sup = [c for c in df_terrazas.columns if 'Superficie' in c]
for c in cols_sup:
    df_terrazas[c] = pd.to_numeric(
        df_terrazas[c].astype(str).str.replace(',', '.', regex=False).str.strip(),
        errors='coerce'
    ).fillna(0)

# Eliminamos filas que no tengan ningun sentido
df_licencias.dropna(how='all', inplace=True)

## 4. Exportación Final Curada

Con todo esto solucionado, forcé la escritura en `utf-8` para que Dremio pudiera comerse el dataset sin corromperse por las tildes madrileñas.

In [ ]:
df_terrazas.to_csv(os.path.join(OUT_DIR, "Terrazas_202104_clean.csv"), index=False, sep=';', encoding='utf-8')
df_licencias.to_csv(os.path.join(OUT_DIR, "Licencias_Locales_202104_clean.csv"), index=False, sep=';', encoding='utf-8')

print(f"Archivos exportados exitosamente en la capeta: {OUT_DIR}")